In [ ]:
import csv
from pathlib import Path
from typing import List
import pandas as pd
from __future__ import annotations
from collections import Counter, defaultdict

def find_root_by_folder_name(folder_name: str, start: Path | None = None) -> Path:
    """
    Find the nearest ancestor directory whose name matches `folder_name`.

    Works when code is run from different subdirectories inside the project.
    """
    if start is None:
        # Use __file__ in scripts; fall back to cwd in notebooks/interpreters
        try:
            start = Path(__file__).resolve()
        except NameError:
            start = Path.cwd().resolve()
    else:
        start = Path(start).resolve()

    # If start is a file, begin from its parent
    if start.is_file():
        start = start.parent

    for path in [start, *start.parents]:
        if path.name == folder_name:
            return path

    raise FileNotFoundError(
        f"Could not find a parent directory named {folder_name!r} from {start}"
    )


ROOT_DIR = find_root_by_folder_name("filler_gap_detector_childes")
STATS_DIR = ROOT_DIR / "childes_statistics"

# Collect Corpus/Transcription-level Information

Before running these cell, run `childes-db.R`, which saves `All_Transcripts.csv` and `All_Utterances.csv` into `ROOT_DIR/childes_statistics`.
- `All_Transcripts.csv` contains the metadata for each transcript from CHILDES, extracted via `childes-db`.
- `All_Utterances.csv` contains the metadata for each utterance from CHILDES, extracted via `childes-db`.

## 1. Create Transcript-level Meta Data

In [9]:
# Process the raw transcripts CSV to filter for English North American data and drop unnecessary columns
df = pd.read_csv(f"{STATS_DIR}/All_Transcripts.csv")
df = df[df['collection_name'] == 'Eng-NA']
assert df['corpus_name'].nunique() == 57, f"Expected 57 unique corpora, found {df['corpus_name'].nunique()}"
df = df.drop(columns=['language', 'date', 'target_child_sex', 'collection_name', 'pid', 'collection_id', 'corpus_id', 'target_child_id'])
df.to_csv(f"{STATS_DIR}/Processed_Transcripts.csv", index=False)

## 2. Annotate Transcripts with Study Type and Activity

In [ ]:
TRANSCRIPT_PATH = Path(f"{STATS_DIR}/Processed_Transcripts.csv")  
CORPUS_DIR = Path(f"{ROOT_DIR}/datasets/CHILDES-Eng-NA")                 

def to_cha_path(filename: str) -> Path:
    """
    Convert the CSV's filename like 'Eng-NA/Garvey/amyann.xml'
    to a local path like 'CHILDES-Eng-NA/Garvey/amyann.cha'
    """
    # 1) .xml -> .cha
    rel = filename.replace(".xml", ".cha")

    # 2) swap root 'Eng-NA/' -> 'CHILDES-Eng-NA/'
    if rel.startswith("Eng-NA/") or rel.startswith("Eng-NA\\"):
        rel = rel.replace("Eng-NA", str(CORPUS_DIR), 1)
        return Path(rel)

    # If it doesn't start with Eng-NA, treat it as relative under CORPUS_DIR
    return CORPUS_DIR / rel

def parse_types_file(types_path: Path) -> tuple[str, str]:
    """
    Parse 0types.txt first non-empty line:
      @Types: cross, toyplay, TD
    Return (study_type, activity) = ("cross", "toyplay")
    """
    with types_path.open("r", encoding="utf-8", errors="replace") as f:
        nonempty = [ln.strip() for ln in f if ln.strip()]

    if not nonempty:
        raise RuntimeError(f"Empty 0types.txt: {types_path}")

    line = nonempty[0]
    if not line.startswith("@Types:"):
        raise RuntimeError(f"Bad 0types.txt header in {types_path}: {line!r}")

    payload = line[len("@Types:"):].strip()
    parts = [p.strip() for p in payload.split(",") if p.strip()]
    if len(parts) < 2:
        raise RuntimeError(f"Expected at least 2 type items in {types_path}, got: {parts}")

    study_type = parts[0]
    activity = parts[1]
    return study_type, activity

def find_nearest_types_file(cha_path: Path, corpus_name: str) -> Path:
    """
    Search for 0types.txt starting in cha_path.parent, then parent, ...,
    until reaching the corpus root (CORPUS_DIR / corpus_name). Include corpus root.
    """
    corpus_root = (CORPUS_DIR / corpus_name).resolve()
    cur = cha_path.parent.resolve()

    # Walk upward until we pass corpus_root
    while True:
        candidate = cur / "0types.txt"
        if candidate.is_file():
            return candidate

        if cur == corpus_root:
            break
        if corpus_root not in cur.parents and cur != corpus_root:
            # cha_path isn't under the expected corpus root; still walk up until filesystem root
            # but this usually signals a path mismatch.
            pass

        # stop at filesystem root
        if cur.parent == cur:
            break
        cur = cur.parent

    raise FileNotFoundError(f"Could not find 0types.txt for {cha_path} (corpus={corpus_name}) up to {corpus_root}")

df = pd.read_csv(TRANSCRIPT_PATH)

types_cache: dict[Path, tuple[str, str]] = {}
study_types = []
activities = []

for _, row in df.iterrows():
    filename = row["filename"]
    corpus_name = row["corpus_name"]

    cha_path = to_cha_path(str(filename))
    types_path = find_nearest_types_file(cha_path, str(corpus_name))

    if types_path not in types_cache:
        types_cache[types_path] = parse_types_file(types_path)

    st, act = types_cache[types_path]
    study_types.append(st)
    activities.append(act)

df["study_type"] = study_types
df["activity"] = activities

# Save back to the original file
df.to_csv(TRANSCRIPT_PATH, index=False)
print(f"Updated {TRANSCRIPT_PATH} with columns: study_type, activity")
print(f"Unique 0types.txt files used: {len(types_cache)}")

Updated /Users/herbertzhou/Documents/Projects/childes_fgd_stats_clean/childes_statistics/Processed_Transcripts.csv with columns: study_type, activity
Unique 0types.txt files used: 104


In [12]:
df = pd.read_csv(TRANSCRIPT_PATH)
df['activity'].value_counts()

activity
toyplay      6096
preverbal    1610
narrative     839
adult         432
meal          314
tests         311
book          292
everyday      197
writing       125
group         122
reading        32
pictures       26
Name: count, dtype: int64

## 3. Inspect Speaker Roles and Age Metadata

In [7]:
# Count of non-Target_Child @ID lines with age present
role_counts = Counter()

# For mean age (months) per role (excluding Target_Child), only when age present
role_age_sum = defaultdict(int)   # role -> sum(age_months)
role_age_n = defaultdict(int)     # role -> n

total_id_lines = 0

# Target_Child counts
total_target_child_ids = 0
total_target_child_with_age = 0

# Child counts
total_child_role_ids = 0
total_child_role_with_age = 0

# All non-target roles with age present
total_non_target_with_age = 0

# Rows for Child-role-with-age CSV
child_role_age_rows = []  # dicts: filename, corpus_name, child_code, age(months)

def age_to_months(age: str, *, path: Path, line_no: int) -> int:
    """
    Convert CHAT age string to months.
      - "2;11.07" -> 2*12 + 11
      - "2;04."   -> 2*12 + 4
      - "2;4"     -> 2*12 + 4
      - "2;"      -> 2*12 + 0
      - "34;"     -> 34*12 + 0  (adult years)
    Raises if format is unexpected.
    """
    age = age.strip()
    if age == "":
        raise RuntimeError(f"age_to_months called on empty age in {path} at line {line_no}")

    if ";" not in age:
        raise RuntimeError(f"Unexpected age format (missing ';') in {path} at line {line_no}: {age!r}")

    years_str, rest = age.split(";", 1)
    years_str = years_str.strip()
    if years_str == "" or not years_str.isdigit():
        raise RuntimeError(f"Unexpected age format (bad years) in {path} at line {line_no}: {age!r}")

    years = int(years_str)

    # rest may be "", "03.", "11.07", "4", etc.
    rest = rest.strip()
    if rest == "":
        months = 0
    else:
        # take up to first '.' if present
        months_str = rest.split(".", 1)[0].strip()
        if months_str == "":
            months = 0
        else:
            if not months_str.isdigit():
                raise RuntimeError(f"Unexpected age format (bad months) in {path} at line {line_no}: {age!r}")
            months = int(months_str)

    return years * 12 + months


for cha_path in CORPUS_DIR.rglob("*.cha"):
    with cha_path.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, raw in enumerate(f, start=1):
            if not raw.startswith("@ID:"):
                continue

            total_id_lines += 1
            line = raw.rstrip("\n")

            # Extract payload after @ID: (prefer tab, fall back to whitespace)
            if "\t" in line:
                payload = line.split("\t", 1)[1]
            else:
                parts = line.split(None, 1)
                if len(parts) < 2:
                    raise RuntimeError(f"Malformed @ID line (no payload) in {cha_path} at line {line_no}: {line!r}")
                payload = parts[1].strip()

            fields = payload.split("|")  # keep empty fields
            if len(fields) < 8:
                raise RuntimeError(
                    f"Malformed @ID payload (expected >= 8 fields) in {cha_path} at line {line_no}:\n"
                    f"  payload={payload!r}\n  fields={fields}"
                )

            corpus_name = fields[1].strip()
            child_code = fields[2].strip()
            age_str = fields[3].strip()
            role = fields[7].strip() or "(EMPTY_ROLE)"

            # --- Target_Child counts (total + with age) ---
            if role == "Target_Child":
                total_target_child_ids += 1
                if age_str != "":
                    total_target_child_with_age += 1
                # Do NOT include Target_Child in "other roles" stats
                continue

            # --- Child-role counts (total + with age) ---
            if role == "Child":
                total_child_role_ids += 1
                if age_str != "":
                    total_child_role_with_age += 1

            # --- Non-Target roles with age: count + mean age ---
            if age_str != "":
                age_months = age_to_months(age_str, path=cha_path, line_no=line_no)

                total_non_target_with_age += 1
                role_counts[role] += 1
                role_age_sum[role] += age_months
                role_age_n[role] += 1

                # Special export: Child role with age
                if role == "Child":
                    rel = cha_path.relative_to(CORPUS_DIR)
                    filename_out = str(Path(CORPUS_DIR.name) / rel)  # "CHILDES-Eng-NA/...."
                    child_role_age_rows.append(
                        {
                            "filename": filename_out,
                            "corpus_name": corpus_name,
                            "child_code": child_code,
                            "age": age_months,
                        }
                    )

# ----------------------
# PRINT REPORT
# ----------------------
print("Scan summary")
print(f"  Total @ID lines:                    {total_id_lines}")
print(f"  Target_Child @ID lines:             {total_target_child_ids}")
print(f"  Non-Target_Child @ID with AGE:      {total_non_target_with_age}")

print("\nTarget_Child-role @ID lines:")
print(f"  Total role=='Target_Child':         {total_target_child_ids}")
print(f"  role=='Target_Child' with AGE:      {total_target_child_with_age}")

print("\nChild-role @ID lines:")
print(f"  Total role=='Child':                {total_child_role_ids}")
print(f"  role=='Child' with AGE filled:      {total_child_role_with_age}")

print("\nRoles (excluding Target_Child) that have age filled: count + mean age (months)")
for role, cnt in role_counts.most_common():
    mean_months = role_age_sum[role] / role_age_n[role]
    print(f"  {role}: {cnt}  (mean_age_months={mean_months:.2f})")

# ----------------------
# WRITE CHILD ROLE CSV
# ----------------------
out_child_csv = Path(f"{STATS_DIR}/child_role_with_age.csv").resolve()
with out_child_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "corpus_name", "child_code", "age"])
    writer.writeheader()
    writer.writerows(child_role_age_rows)

print(f"\nWrote {len(child_role_age_rows)} rows to {out_child_csv}")

Scan summary
  Total @ID lines:                    26838
  Target_Child @ID lines:             10096
  Non-Target_Child @ID with AGE:      1035

Target_Child-role @ID lines:
  Total role=='Target_Child':         10096
  role=='Target_Child' with AGE:      9349

Child-role @ID lines:
  Total role=='Child':                747
  role=='Child' with AGE filled:      170

Roles (excluding Target_Child) that have age filled: count + mean age (months)
  Mother: 359  (mean_age_months=312.06)
  Investigator: 173  (mean_age_months=312.72)
  Child: 170  (mean_age_months=43.02)
  Father: 129  (mean_age_months=313.40)
  Media: 58  (mean_age_months=310.34)
  Adult: 35  (mean_age_months=311.66)
  Grandmother: 31  (mean_age_months=609.68)
  Brother: 27  (mean_age_months=140.89)
  Environment: 25  (mean_age_months=307.20)
  Sister: 8  (mean_age_months=126.38)
  Unidentified: 8  (mean_age_months=312.00)
  Friend: 5  (mean_age_months=261.60)
  Visitor: 4  (mean_age_months=312.00)
  Grandfather: 2  (mean_a

## 4. Export Known Non-Target Child Ages & Link Non-Target Child Ages to Transcript IDs

In [8]:
child_csv = f"{STATS_DIR}/child_role_with_age.csv"
transcripts_csv = f"{STATS_DIR}/Processed_Transcripts.csv"

def normalize_path_str(p: str) -> str:
    """Force forward slashes + strip whitespace."""
    return str(p).strip().replace("\\", "/")

def cha_childes_to_key(p: str) -> str:
    """
    child_role_with_age.csv has filename like:
      CHILDES-Eng-NA/Providence/Alex/011006.cha

    Convert to a canonical key like:
      Eng-NA/Providence/Alex/011006.xml
    """
    p = normalize_path_str(p)
    # replace root folder
    if p.startswith("CHILDES-Eng-NA/"):
        p = "Eng-NA/" + p[len("CHILDES-Eng-NA/"):]
    # replace extension
    if p.endswith(".cha"):
        p = p[:-4] + ".xml"
    return p

def xml_engna_to_key(p: str) -> str:
    """
    transcripts csv 'filename' usually like:
      Eng-NA/Providence/Alex/011006.xml
    Return canonical key in same form.
    """
    p = normalize_path_str(p)
    # ensure root is Eng-NA (some exports might include CHILDES-Eng-NA)
    if p.startswith("CHILDES-Eng-NA/"):
        p = "Eng-NA/" + p[len("CHILDES-Eng-NA/"):]
    # ensure .xml (some might be .cha)
    if p.endswith(".cha"):
        p = p[:-4] + ".xml"
    return p

# --------------------
# Load
# --------------------
child_df = pd.read_csv(child_csv)
tx_df = pd.read_csv(transcripts_csv)

# --------------------
# Build match keys
# --------------------
child_df["_key"] = child_df["filename"].map(cha_childes_to_key)
tx_df["_key"] = tx_df["filename"].map(xml_engna_to_key)

# Build lookup from key -> transcript_id, but enforce uniqueness
dup_keys = tx_df["_key"][tx_df["_key"].duplicated(keep=False)]
if len(dup_keys) > 0:
    # Show a few duplicates to debug
    example = tx_df[tx_df["_key"].isin(dup_keys)].sort_values("_key").head(20)
    raise RuntimeError(
        f"Transcripts file has non-unique filename keys (expected unique). "
        f"Examples:\n{example[['filename','transcript_id','_key']].to_string(index=False)}"
    )

tx_map = dict(zip(tx_df["_key"], tx_df["transcript_id"]))

# --------------------
# Attach transcript_id with per-row assertion
# --------------------
missing = child_df[~child_df["_key"].isin(tx_map.keys())]
if len(missing) > 0:
    ex = missing[["filename", "_key"]].head(20)
    raise RuntimeError(
        f"Some child_role_with_age rows have NO transcript match in transcripts CSV. "
        f"Examples:\n{ex.to_string(index=False)}"
    )

child_df["transcript_id"] = child_df["_key"].map(tx_map)

# Final assert: every row got exactly one transcript_id (no NA)
assert child_df["transcript_id"].notna().all()

# Cleanup and save back to same file
child_df = child_df.drop(columns=["_key"])
child_df.to_csv(child_csv, index=False)

print(f"Updated {child_csv} with transcript_id (rows={len(child_df)})")

Updated /Users/herbertzhou/Documents/Projects/childes_fgd_stats_clean/childes_statistics/child_role_with_age.csv with transcript_id (rows=170)


## 5. Correct Utterance-Level Ages for `Child` Speakers

In [ ]:
utts = pd.read_csv(f"{STATS_DIR}/All_Utterances.csv")
child_age = pd.read_csv(f"{STATS_DIR}/child_role_with_age.csv")

for idx, row in child_age.iterrows():
    tx_id = row['transcript_id']
    child_code = row['child_code']
    mask = (utts['transcript_id'] == tx_id) & (utts['speaker_code'] == child_code)
    utts.loc[mask, 'target_child_age'] = row['age']

# only keep relevant columns
utts = utts[['id', 'gloss', 'type', 'corpus_name', 'speaker_code', 'speaker_role', 'target_child_age', 'transcript_id']]
utts.to_csv(f"{STATS_DIR}/Processed_Utterances.csv", index=False)

/var/folders/rd/jz160lxn3g5bc3cyzjv46lmr0000gp/T/ipykernel_15109/525067169.py:1: DtypeWarning: Columns (2,3,4,11,13,15,20) have mixed types. Specify dtype option on import or set low_memory=False.
  utts = pd.read_csv("All_Utterances.csv")


In [ ]:
utts = pd.read_csv(f"{STATS_DIR}/Processed_Utterances.csv")
# calculate the porportion of utterances with target_child_age filled
total_utts = len(utts)
utts_with_age = utts['target_child_age'].notna().sum()
proportion_with_age = utts_with_age / total_utts
print(f"Proportion of utterances with target_child_age filled: {proportion_with_age:.4f} ({utts_with_age}/{total_utts})")

Proportion of utterances with target_child_age filled: 0.9190 (2935874/3194544)


## 6. Fill Remaining Missing Ages from CHAT `@ID` Lines

In [13]:
from __future__ import annotations

import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, Optional

# -----------------------
# CONFIG
# -----------------------
UTTS_PATH = Path(f"{STATS_DIR}/Processed_Utterances.csv")
TX_PATH   = Path(f"{STATS_DIR}/Processed_Transcripts.csv")

LOCAL_ROOT = Path(f"{ROOT_DIR}/CHILDES-Eng-NA")   # your local folder name
OUT_PATH = Path(f"{STATS_DIR}/Processed_Utterances_Updated.csv")  # follow your requested filename

CHILD_ROLES = {"Child", "Target_Child"}  # roles treated as "child-like"
TARGET_ROLE = "Target_Child"


# -----------------------
# Helpers
# -----------------------
def normalize_role(role: str) -> str:
    """Normalize role strings like 'Target Child' -> 'Target_Child'."""
    if pd.isna(role):
        return ""
    role = str(role).strip()
    role = role.replace(" ", "_")
    return role

def age_to_months(age: str) -> Optional[int]:
    """
    Convert CHAT age string -> months.
      '2;11.07' -> 2*12 + 11
      '2;04.'   -> 2*12 + 4
      '2;4'     -> 2*12 + 4
      '2;'      -> 2*12 + 0
      '34;'     -> 34*12 + 0
    Return None if empty or unparseable.
    """
    if age is None:
        return None
    age = str(age).strip()
    if age == "" or age.lower() == "nan":
        return None
    if ";" not in age:
        return None
    y_str, rest = age.split(";", 1)
    y_str = y_str.strip()
    if not y_str.isdigit():
        return None
    years = int(y_str)

    rest = rest.strip()
    if rest == "":
        months = 0
    else:
        m_str = rest.split(".", 1)[0].strip()
        months = int(m_str) if m_str.isdigit() else 0

    return years * 12 + months

def transcript_xml_to_local_cha(xml_path: str) -> Path:
    """
    Convert 'Eng-NA/.../foo.xml' to 'CHILDES-Eng-NA/.../foo.cha'
    """
    p = str(xml_path).strip().replace("\\", "/")
    if p.startswith("Eng-NA/"):
        p = str(LOCAL_ROOT) + "/" + p[len("Eng-NA/"):]
    else:
        # if already relative without Eng-NA, assume under LOCAL_ROOT
        p = str(LOCAL_ROOT) + "/" + p.lstrip("/")
    if p.endswith(".xml"):
        p = p[:-4] + ".cha"
    return Path(p)

def parse_id_lines(cha_path: Path) -> Dict[Tuple[str, str], Optional[int]]:
    """
    Parse all @ID lines in a .cha file.
    Return a mapping: (speaker_code, speaker_role) -> age_months_or_None

    Also normalizes role to underscores.
    """
    id_map: Dict[Tuple[str, str], Optional[int]] = {}
    # Read with replace to avoid decode issues
    with cha_path.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            if not raw.startswith("@ID:"):
                continue

            line = raw.rstrip("\n")
            # payload after tab preferred
            if "\t" in line:
                payload = line.split("\t", 1)[1]
            else:
                parts = line.split(None, 1)
                if len(parts) < 2:
                    continue
                payload = parts[1].strip()

            fields = payload.split("|")
            if len(fields) < 8:
                continue

            code = fields[2].strip()
            age_str = fields[3].strip()
            role = normalize_role(fields[7].strip())

            id_map[(code, role)] = age_to_months(age_str)

    return id_map

def get_target_child_age(id_map: Dict[Tuple[str, str], Optional[int]]) -> Optional[int]:
    """
    Get an age for Target_Child, even if we don't know the code.
    Prefer any non-None.
    """
    for (code, role), age_m in id_map.items():
        if role == TARGET_ROLE and age_m is not None:
            return age_m
    # If all Target_Child are None, return None
    return None


# -----------------------
# Load data
# -----------------------
utts = pd.read_csv(UTTS_PATH)
tx = pd.read_csv(TX_PATH)

# Normalize roles in utts (handles "Target Child" vs "Target_Child")
utts["speaker_role"] = utts["speaker_role"].map(normalize_role)

# Ensure target_child_age is numeric-ish, but keep NaN
utts["target_child_age"] = pd.to_numeric(utts["target_child_age"], errors="coerce")

# Build transcript_id -> filename map
if "transcript_id" not in tx.columns or "filename" not in tx.columns:
    raise RuntimeError("Processed_Transcripts.csv must contain columns: transcript_id, filename")

tx_map = dict(zip(tx["transcript_id"], tx["filename"]))

# Only work on rows with missing target_child_age
missing_mask = utts["target_child_age"].isna()
missing_rows = utts.loc[missing_mask, ["transcript_id"]].drop_duplicates()

# Cache parsed ID maps per transcript_id
id_cache: Dict[int, Dict[Tuple[str, str], Optional[int]]] = {}
target_age_cache: Dict[int, Optional[int]] = {}

# Pre-parse only the transcripts we need
for tx_id in missing_rows["transcript_id"].tolist():
    if tx_id not in tx_map:
        # transcript_id not in transcripts table; can't fill
        id_cache[tx_id] = {}
        target_age_cache[tx_id] = None
        continue

    cha_path = transcript_xml_to_local_cha(tx_map[tx_id])
    if not cha_path.is_file():
        id_cache[tx_id] = {}
        target_age_cache[tx_id] = None
        continue

    id_map = parse_id_lines(cha_path)
    id_cache[tx_id] = id_map
    target_age_cache[tx_id] = get_target_child_age(id_map)

# -----------------------
# Fill missing ages
# -----------------------
def infer_age_for_row(row) -> Optional[int]:
    """
    Implements your logic for a single utterance row.
    Returns age in months or None.
    """
    tx_id = row["transcript_id"]
    spk_code = str(row["speaker_code"])
    spk_role = normalize_role(row["speaker_role"])

    id_map = id_cache.get(tx_id, {})
    target_age = target_age_cache.get(tx_id, None)

    # 4.1 If speaker_role is NOT in {Child, Target_Child}: use target child's age
    if spk_role not in CHILD_ROLES:
        return target_age

    # 4.2 If speaker_role is Child: try that child's age; if missing, fallback to target child's age
    if spk_role == "Child":
        child_age = id_map.get((spk_code, "Child"), None)
        if child_age is not None:
            return child_age
        # If child's age missing/unavailable, fallback to target child age if available
        return target_age

    # 4.3 If speaker_role is Target_Child: try matching by code+role; if missing, fallback to any Target_Child
    if spk_role == "Target_Child":
        tc_age = id_map.get((spk_code, "Target_Child"), None)
        if tc_age is not None:
            return tc_age
        return target_age

    # Shouldn't happen given CHILD_ROLES, but safe default
    return target_age

# Only apply to missing rows
utts.loc[missing_mask, "target_child_age"] = utts.loc[missing_mask].apply(infer_age_for_row, axis=1)

# -----------------------
# Save
# -----------------------
utts.to_csv(OUT_PATH, index=False)
print(f"Saved updated utterances to: {OUT_PATH}")
print(f"Filled count: {(missing_mask & utts['target_child_age'].notna()).sum()} / {missing_mask.sum()} originally-missing rows")


Saved updated utterances to: /Users/herbertzhou/Documents/Projects/childes_fgd_stats_clean/childes_statistics/Processed_Utterances_Updated.csv
Filled count: 144187 / 258670 originally-missing rows


## 7. Preliminary Checks Before Statistical Analysis

### Check the activities to define naturalistic production vs. {story reading, writing, experiment, etc.}

In [ ]:
df = pd.read_csv("Processed_Transcripts_Updated.csv")
df['activity'].value_counts()

activity
toyplay      6096
preverbal    1610
narrative     839
adult         432
meal          314
tests         311
book          292
everyday      197
writing       125
group         122
reading        32
pictures       26
Name: count, dtype: int64

### Sentence type

In [2]:
df = pd.read_csv("Processed_Utterances_Updated.csv")
df['type'].value_counts()

type
declarative                   2267039
question                       695923
imperative_emphatic            135196
trail off                       60858
interruption                    13409
self interruption                9881
quotation next line              4919
missing CA terminator            3969
trail off question               1264
self interruption question        929
interruption question             589
quotation precedes                551
broken for coding                  14
question exclamation                3
Name: count, dtype: int64

In [19]:
for sen in df[df['type'] == 'trail off'].head(100)['gloss'].tolist():
    # if len(sen) > 30 or len(sen) < 15:
    print(sen)

that's e shoe
and that
now we need this
I'm I'm the
now we need
oh
food
now we needta cook this snake this
that's not
I brought wha
aw you're ss you're
okay
there's no pla
you can
oh for heaven's sake here's the bowl this is the pan here's the bowl now we should add
so I'll get the get the
not yet put
I got a
he's a
okay need another xxx
we hafta get all our cl
we cook our treasures tend I'm the mother um um um I live across the street and you're you're
yeah and this
and pretend this is
okay
and we hafta put
honey I know I could get cha
hey can xxx
yes he will he will
what
no wait a minute you can have the scr
I say hi Bubba
pretend that I
what kinda no
your measuring
hi mom
hi mom
oo it's feathers on it oh I found
I found a flash
it's uh his name is probably
no it's poo poo daigi and dia
that's not not
I don't
hey little chair
what're you doing
now I
this one's yours
there's so m
yeah that's
yeah so xxx where's the fire engine
should I put this seat up here I would be driving
and it h

In [ ]:
sentence_types = df['type'].unique().tolist()
print(sentence_types)

# manually assign final punctuation marks for each type
sentence_types = {'declarative': ".",
                  'question': "?", 
                  'trail off': " ...", 
                  'interruption': "", 
                  'trail off question': "?", 
                  'imperative_emphatic': "!",
                  'interruption question': "?", 
                  'quotation next line': ":",
                  'self interruption': "...", 
                  'quotation precedes': ".", 
                  'self interruption question': " xxx?", 
                  'broken for coding': ".", 
                  'missing CA terminator': "", 
                  'question exclamation': "!"}

['declarative', 'question', 'trail off', 'interruption', 'trail off question', 'imperative_emphatic', 'interruption question', 'quotation next line', 'self interruption', 'quotation precedes', 'self interruption question', 'broken for coding', 'missing CA terminator', 'question exclamation']


## Bonus: (Dis)aggregation Helper Functions

In [50]:
# get the rows where the "target_child_age" is nan, and see the total number of unique "transcript_id"s involved
missing_age_rows = utts[utts['target_child_age'].isna()]
unique_transcript_ids = missing_age_rows['transcript_id'].nunique()
print(f"Number of unique transcript_ids with missing target_child_age: {unique_transcript_ids}")

Number of unique transcript_ids with missing target_child_age: 670
